# AQI Predictor

This project predicts the AQI for the next 3 days in Karachi, Lahore, Faisalabad, Islamabad and Peshawar. Pollutants come from OpenWeather and weather comes from Open-Meteo. The features are stored in Hopsworks and the forecast is shown on a Streamlit dashboard.

## 00 — Setup

I will set up the constants used in the rest of the notebook: the five cities, the API endpoints and the EPA breakpoint table that turns pollutant readings into an AQI number.

In [1]:
import os
import pathlib
from datetime import datetime,timedelta,timezone
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from tenacity import retry,stop_after_attempt,wait_exponential

load_dotenv()
OPENWEATHER_KEY = os.getenv("OPENWEATHER_API_KEY")
HOPSWORKS_KEY = os.getenv("HOPSWORKS_API_KEY")

data_dir = pathlib.Path("data")
fig_dir = pathlib.Path("outputs/figures")
data_dir.mkdir(parents=True,exist_ok=True)
fig_dir.mkdir(parents=True,exist_ok=True)

sns.set_theme(style="whitegrid")

### Cities

I will use the city centre coordinates for the five cities. Both APIs take a lat and lon so the same dictionary works for both.

In [2]:
CITIES = {
    "Karachi":(24.86,67.01),
    "Lahore":(31.55,74.35),
    "Faisalabad":(31.41,73.07),
    "Islamabad":(33.72,73.06),
    "Peshawar":(34.01,71.57),
}

### API Endpoints

OpenWeather only sells historical weather on a paid plan so I will take the pollutants from OpenWeather and the weather from Open-Meteo which gives the ERA5 archive for free and without a key. Open-Meteo documents a lag of a few days on the archive endpoint so I have kept the forecast endpoint here as a fallback and will confirm at Step 1 whether the hourly pipeline actually needs it.

In [3]:
POLLUTION_URL = "http://api.openweathermap.org/data/2.5/air_pollution"
POLLUTION_HISTORY_URL = "http://api.openweathermap.org/data/2.5/air_pollution/history"
POLLUTION_FORECAST_URL = "http://api.openweathermap.org/data/2.5/air_pollution/forecast"

WEATHER_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

POLLUTANTS = ["pm2_5","pm10","o3","no2","so2","co","no","nh3"]

WEATHER_VARS = [
    "temperature_2m","relative_humidity_2m","dew_point_2m","precipitation",
    "surface_pressure","wind_speed_10m","wind_direction_10m","cloud_cover",
]

### EPA Breakpoints

OpenWeather returns its own AQI but it is only an integer from 1 to 5 which is too coarse to regress on. So I will compute the US EPA AQI instead, the 0 to 500 scale that aqicn and most public dashboards show.

The EPA gives a breakpoint table for each pollutant. Each row maps a concentration range to an AQI range and the AQI is found by interpolating between the two. Each pollutant is also averaged over a set number of hours before the lookup and the four gases have to be converted from µg/m³ first, so I will keep those here too.

In [4]:
# conc low, conc high, aqi low, aqi high
BREAKPOINTS = {
    # pm2_5 is the 2024 revised table
    "pm2_5":[
        (0.0,9.0,0,50),(9.1,35.4,51,100),(35.5,55.4,101,150),
        (55.5,125.4,151,200),(125.5,225.4,201,300),(225.5,325.4,301,500),
    ],
    "pm10":[
        (0,54,0,50),(55,154,51,100),(155,254,101,150),
        (255,354,151,200),(355,424,201,300),(425,604,301,500),
    ],
    "o3":[
        (0.000,0.054,0,50),(0.055,0.070,51,100),(0.071,0.085,101,150),
        (0.086,0.105,151,200),(0.106,0.200,201,300),
    ],
    "co":[
        (0.0,4.4,0,50),(4.5,9.4,51,100),(9.5,12.4,101,150),
        (12.5,15.4,151,200),(15.5,30.4,201,300),(30.5,50.4,301,500),
    ],
    "so2":[
        (0,35,0,50),(36,75,51,100),(76,185,101,150),
        (186,304,151,200),(305,604,201,300),(605,1004,301,500),
    ],
    "no2":[
        (0,53,0,50),(54,100,51,100),(101,360,101,150),
        (361,649,151,200),(650,1249,201,300),(1250,2049,301,500),
    ],
}

AVERAGING_HOURS = {"pm2_5":24,"pm10":24,"o3":8,"co":8,"so2":1,"no2":1}

# o3 and co need ppm and the other two need ppb
MOLECULAR_WEIGHTS = {"o3":48.0,"co":28.01,"so2":64.06,"no2":46.01}

### AQI Categories

The scale is split into six named bands. I will store each band with its upper limit so I can find the category for any AQI value.

In [5]:
AQI_CATEGORIES = [
    (50,"Good","green"),
    (100,"Moderate","gold"),
    (150,"Unhealthy for Sensitive Groups","orange"),
    (200,"Unhealthy","red"),
    (300,"Very Unhealthy","purple"),
    (500,"Hazardous","darkred"),
]

## 01 — Data Fetch

I will write fetch_pollution and fetch_weather and test both on Lahore over the last 7 days so I can check the shapes, units and timestamp alignment before pulling the full three year history.

Outputs:
- pollution_test — hourly pollutant components for Lahore, last 7 days
- weather_test — hourly weather variables for Lahore, last 7 days

### Setup

In [6]:
lat,lon = CITIES["Lahore"]

end = datetime.now(timezone.utc)
start = end - timedelta(days=7)

### Pollution

The history endpoint takes start and end as unix timestamps in UTC so I will convert the datetime objects before calling it.

In [7]:
@retry(stop=stop_after_attempt(3),wait=wait_exponential(min=2,max=10))
def fetch_pollution(lat,lon,start,end):
    r = requests.get(POLLUTION_HISTORY_URL,params={"lat":lat,"lon":lon,"start":int(start.timestamp()),"end":int(end.timestamp()),"appid":OPENWEATHER_KEY},timeout=30)
    r.raise_for_status()
    return r.json()

data = fetch_pollution(lat,lon,start,end)
rows = []
for item in data["list"]:
    row = item["components"]
    row["timestamp_utc"] = pd.to_datetime(item["dt"],unit="s",utc=True)
    rows.append(row)
pollution_test = pd.DataFrame(rows)
print(f"lahore pollution: {pollution_test.shape}")
pollution_test.head()

lahore pollution: (168, 9)


,co,no,no2,o3,so2,pm2_5,pm10,nh3,timestamp_utc
0,136.26,0.01,3.92,73.37,3.21,13.80,47.10,7.47,2026-07-20 13:00:00+00:00
1,174.62,0.00,6.39,63.56,3.66,14.55,47.03,11.08,2026-07-20 14:00:00+00:00
2,216.98,0.00,8.59,54.51,4.12,15.74,45.73,14.31,2026-07-20 15:00:00+00:00
3,262.37,0.00,10.24,46.46,4.84,18.32,43.02,16.04,2026-07-20 16:00:00+00:00
4,316.65,0.00,11.62,41.14,6.02,22.33,41.39,16.35,2026-07-20 17:00:00+00:00


### Weather

I will pull the same window from the weather archive.

In [8]:
@retry(stop=stop_after_attempt(3),wait=wait_exponential(min=2,max=10))
def fetch_weather(lat,lon,start,end):
    r = requests.get(WEATHER_ARCHIVE_URL,params={"latitude":lat,"longitude":lon,"start_date":start.strftime("%Y-%m-%d"),"end_date":end.strftime("%Y-%m-%d"),"hourly":",".join(WEATHER_VARS),"timezone":"UTC"},timeout=30)
    r.raise_for_status()
    return r.json()

data = fetch_weather(lat,lon,start,end)
weather_test = pd.DataFrame(data["hourly"])
weather_test["time"] = pd.to_datetime(weather_test["time"],utc=True)
print(f"lahore weather: {weather_test.shape}")
weather_test.head()

lahore weather: (192, 9)


,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,cloud_cover
0,2026-07-20 00:00:00+00:00,31.0,77,26.4,0.0,974.9,8.7,187,96
1,2026-07-20 01:00:00+00:00,31.1,76,26.3,0.0,974.8,4.7,148,90
2,2026-07-20 02:00:00+00:00,31.7,74,26.5,0.0,975.8,8.7,84,98
3,2026-07-20 03:00:00+00:00,31.3,78,27.0,0.2,975.8,9.1,126,100
4,2026-07-20 04:00:00+00:00,33.2,69,26.8,0.0,975.8,10.7,131,97


## 02 — EPA AQI

I will average each pollutant over the window the EPA requires, convert the four gases into the units the breakpoint table expects and interpolate the table to get an AQI value and the pollutant that produced it.

Outputs:
- aqi and dominant_pollutant for the most recent hour in pollution_test

### Averaging

pm2_5 and pm10 use a 24 hour rolling average, ozone and carbon monoxide use 8 hours and sulfur dioxide and nitrogen dioxide use 1 hour. I will average each pollutant over its own window from the tail of pollution_test.

In [9]:
latest = {}
for pollutant in AVERAGING_HOURS:
    window = AVERAGING_HOURS[pollutant]
    latest[pollutant] = pollution_test[pollutant].tail(window).mean()

### AQI

The gases in latest are still in µg/m³ so I will convert them with their molecular weight before looking each one up in its breakpoint table. Each row of the table maps a concentration range onto an AQI range and I will interpolate inside whichever row the concentration falls in, then take the pollutant with the highest sub index as the overall AQI.

In [10]:
def convert_units(conc,pollutant):
    if pollutant not in MOLECULAR_WEIGHTS:
        return conc
    ppb = conc * 24.45 / MOLECULAR_WEIGHTS[pollutant]
    if pollutant in ("o3","co"):
        return ppb / 1000
    return ppb

def calc_aqi(conc,pollutant):
    for bp_lo,bp_hi,aqi_lo,aqi_hi in BREAKPOINTS[pollutant]:
        if bp_lo <= conc <= bp_hi:
            return (aqi_hi-aqi_lo) / (bp_hi-bp_lo) * (conc-bp_lo) + aqi_lo

def compute_aqi(concentrations):
    sub_indices = {}
    for pollutant,conc in concentrations.items():
        converted = convert_units(conc,pollutant)
        sub_indices[pollutant] = calc_aqi(converted,pollutant)
    dominant_pollutant = max(sub_indices,key=sub_indices.get)
    aqi = round(sub_indices[dominant_pollutant])
    return aqi,dominant_pollutant

aqi,dominant_pollutant = compute_aqi(latest)
print(f"lahore aqi: {aqi} and dominant pollutant {dominant_pollutant}")

lahore aqi: 155 and dominant pollutant pm2_5
